In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
#O TensorFlow só entende números. Vamos converter os textos em números (0 ou 1)
dataset['smoker'] = dataset['smoker'].map({'yes': 1, 'no': 0})
dataset['sex'] = dataset['sex'].map({'female': 1, 'male': 0})

# A coluna "region" tem 4 regiões diferentes. O 'get_dummies' transforma cada
# região em uma nova coluna de 0 ou 1.
dataset = pd.get_dummies(dataset, columns=['region'])

# Garante que todo o dataset seja numérico (float)
dataset = dataset.astype('float32')

#DIVISÃO 80/20 ---
# Pega 80% dos dados aleatoriamente para treino
train_dataset = dataset.sample(frac=0.8, random_state=42)
# O que sobrar (20%) fica para o teste
test_dataset = dataset.drop(train_dataset.index)

# SEPARANDO O GABARITO (LABELS) ---
# Tiramos a coluna de despesas (expenses) e guardamos como nosso gabarito
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

# NORMALIZAÇÃO ---
# Idade vai de 18 a 60, filhos de 0 a 5. Para a IA não achar que a idade
# vale mais só porque o número é maior, nós normalizamos (equilibramos as escalas).
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(np.array(train_dataset))

#  CRIANDO O CÉREBRO (O MODELO) ---
model = keras.Sequential([
    normalizer, # Primeiro, os dados passam pelo normalizador
    layers.Dense(64, activation='relu'), # Camada oculta para achar padrões complexos
    layers.Dense(64, activation='relu'), # Outra camada oculta
    layers.Dense(1) # A saída: Apenas 1 neurônio que vai cuspir o valor em Dólares
])

#  COMPILANDO E TREINANDO ---
# Usamos o 'mae' (Erro Médio Absoluto) porque é a métrica que o teste exige
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss='mae',
    metrics=['mae', 'mse']
)

# Treinamento: A IA vai estudar os dados por 100 rodadas (epochs)
print("Iniciando o treinamento. Isso pode levar alguns segundos...")
history = model.fit(
    train_dataset,
    train_labels,
    epochs=100,
    validation_split=0.2, # Separa um pedacinho do treino para ver se está indo bem
    verbose=0 # Deixamos 0 para não poluir sua tela com 100 linhas de texto
)
print("Treinamento concluído!")

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
